# Estrategia 1 — EfficientNet B0–B7 + NAS en el clasificador

In [ ]:
# 1. Montar Drive y clonar/actualizar repo
from google.colab import drive
drive.mount('/content/drive')

import os
repo_path = '/content/darts-cards'
if not os.path.exists(repo_path):
    !git clone https://github.com/javiergarciaduran/darts-cards.git {repo_path}
else:
    !git -C {repo_path} pull
%cd {repo_path}

In [ ]:
# 2. Dependencias
!pip install -q torchvision
!pip install -q scikit-learn matplotlib

In [ ]:
# 3. Fix encoding
!iconv -f UTF-16 -t UTF-8 /content/darts-cards/datasets/__init__.py > temp.py && mv temp.py /content/darts-cards/datasets/__init__.py

In [ ]:
# 4. Symlink dataset
!mkdir -p ./data
!ln -sfn /content/drive/MyDrive/cards ./data/cards

# Verificar
from datasets.cards import get_cards
tr, nc = get_cards('./data/cards', input_size=224, split='train')
va, _  = get_cards('./data/cards', input_size=224, split='val')
print(f'train={len(tr)}  val={len(va)}  classes={nc}')

In [ ]:
# 5. Crear carpetas de salida en Drive
!mkdir -p /content/drive/MyDrive/darts_experiments/efficientnet
!mkdir -p /content/drive/MyDrive/darts_logs/efficientnet

## B0 — empezar aquí (el más ligero, ~5 min)
Ejecuta primero B0 para verificar que todo funciona. Luego escala a B1–B7.

In [ ]:
!python efficientnet_search.py \
    --backbone efficientnet_b0 \
    --data_path ./data/cards \
    --path /content/drive/MyDrive/darts_experiments/efficientnet \
    --epochs 50 \
    --batch_size 64 \
    --hidden 256 \
    --input_size 224 \
    --seed 42 \
    2>&1 | tee /content/drive/MyDrive/darts_logs/efficientnet/b0.log

In [ ]:
!python efficientnet_search.py \
    --backbone efficientnet_b1 \
    --data_path ./data/cards \
    --path /content/drive/MyDrive/darts_experiments/efficientnet \
    --epochs 50 \
    --batch_size 64 \
    --hidden 256 \
    --input_size 224 \
    --seed 42 \
    2>&1 | tee /content/drive/MyDrive/darts_logs/efficientnet/b1.log

In [ ]:
!python efficientnet_search.py \
    --backbone efficientnet_b2 \
    --data_path ./data/cards \
    --path /content/drive/MyDrive/darts_experiments/efficientnet \
    --epochs 50 \
    --batch_size 64 \
    --hidden 256 \
    --input_size 224 \
    --seed 42 \
    2>&1 | tee /content/drive/MyDrive/darts_logs/efficientnet/b2.log

In [ ]:
!python efficientnet_search.py \
    --backbone efficientnet_b3 \
    --data_path ./data/cards \
    --path /content/drive/MyDrive/darts_experiments/efficientnet \
    --epochs 50 \
    --batch_size 32 \
    --hidden 256 \
    --input_size 224 \
    --seed 42 \
    2>&1 | tee /content/drive/MyDrive/darts_logs/efficientnet/b3.log

In [ ]:
!python efficientnet_search.py \
    --backbone efficientnet_b4 \
    --data_path ./data/cards \
    --path /content/drive/MyDrive/darts_experiments/efficientnet \
    --epochs 50 \
    --batch_size 32 \
    --hidden 512 \
    --input_size 224 \
    --seed 42 \
    2>&1 | tee /content/drive/MyDrive/darts_logs/efficientnet/b4.log

In [ ]:
!python efficientnet_search.py \
    --backbone efficientnet_b5 \
    --data_path ./data/cards \
    --path /content/drive/MyDrive/darts_experiments/efficientnet \
    --epochs 50 \
    --batch_size 32 \
    --hidden 512 \
    --input_size 224 \
    --seed 42 \
    2>&1 | tee /content/drive/MyDrive/darts_logs/efficientnet/b5.log

In [ ]:
!python efficientnet_search.py \
    --backbone efficientnet_b6 \
    --data_path ./data/cards \
    --path /content/drive/MyDrive/darts_experiments/efficientnet \
    --epochs 50 \
    --batch_size 16 \
    --hidden 512 \
    --input_size 224 \
    --seed 42 \
    2>&1 | tee /content/drive/MyDrive/darts_logs/efficientnet/b6.log

In [ ]:
!python efficientnet_search.py \
    --backbone efficientnet_b7 \
    --data_path ./data/cards \
    --path /content/drive/MyDrive/darts_experiments/efficientnet \
    --epochs 50 \
    --batch_size 16 \
    --hidden 512 \
    --input_size 224 \
    --seed 42 \
    2>&1 | tee /content/drive/MyDrive/darts_logs/efficientnet/b7.log

## Evaluación en test set — ejecutar UNA SOLA VEZ por backbone
Cambia `BACKBONE` según el modelo que quieras evaluar.

In [ ]:
BACKBONE = 'efficientnet_b0'   # cambiar por b1, b2, b3, b4, b5, b6, b7...

!python efficientnet_evaluate.py \
    --checkpoint /content/drive/MyDrive/darts_experiments/efficientnet/{BACKBONE}_seed42/best.pth.tar \
    --data_path ./data/cards \
    --input_size 224

## Comparativa Estrategia 1 vs Estrategia 2

In [ ]:
import os, torch
import matplotlib.pyplot as plt

BASE = '/content/drive/MyDrive/darts_experiments/efficientnet'
backbones = [f'efficientnet_b{i}' for i in range(8)]

results = []
for bb in backbones:
    ckpt_path = os.path.join(BASE, f'{bb}_seed42', 'best.pth.tar')
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        results.append({'backbone': bb, 'val_acc': ckpt['val_acc'],
                        'genotype': ckpt['genotype']})
        print(f"{bb:25s}  val_acc={ckpt['val_acc']:.2f}%  genotype={ckpt['genotype']}")
    else:
        print(f"{bb:25s}  — aún no ejecutado")

# Añadir resultado DARTS (Estrategia 2) para comparar
results.append({'backbone': 'DARTS (Estrategia 2)', 'val_acc': 85.28})

labels  = [r['backbone'].replace('efficientnet_', 'EffNet-') for r in results]
accs    = [r['val_acc'] for r in results]
colors  = ['steelblue'] * (len(results)-1) + ['coral']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, accs, color=colors, edgecolor='white', linewidth=0.8)
ax.bar_label(bars, fmt='%.2f%%', padding=4, fontsize=10)
ax.set_ylim(0, 100)
ax.set_ylabel('Validation Top-1 Accuracy (%)')
ax.set_title('Estrategia 1 (EfficientNet+NAS) vs Estrategia 2 (DARTS)')
ax.axhline(85.28, color='coral', linestyle='--', linewidth=1, label='DARTS baseline')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'comparison.png'), dpi=150)
plt.show()